# SeeSaw — Notebook 3: GGUF Export

Merges LoRA adapters, converts to GGUF float16, quantises to Q4_K_M, validates, and uploads to GCS.

**Input:** `gs://seesaw-models/checkpoints/seesaw-gemma4-v1/`  
**Output:** `gs://seesaw-models/seesaw-gemma4-1b-q4km.gguf` (~800 MB)

**Runtime:** Free Colab T4, ~20 minutes

See `docs/FINE_TUNING.md` for validation criteria.

In [ ]:
!pip install transformers peft google-cloud-storage

In [ ]:
import subprocess

# Clone llama.cpp (shallow)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggerganov/llama.cpp', '/tmp/llama.cpp'], check=True)
subprocess.run(['pip', 'install', '-r', '/tmp/llama.cpp/requirements.txt'], check=True)

# Build quantize tool
subprocess.run(['cmake', '-B', '/tmp/llama.cpp/build', '/tmp/llama.cpp'], check=True)
subprocess.run(['cmake', '--build', '/tmp/llama.cpp/build', '--config', 'Release', '-j4'], check=True)
print('llama.cpp built successfully')

In [ ]:
import subprocess

# Download checkpoint from GCS
GCS_CHECKPOINT = 'gs://seesaw-models/checkpoints/seesaw-gemma4-v1'
LOCAL_MERGED   = '/tmp/seesaw-gemma4-merged'

subprocess.run(['gsutil', '-m', 'cp', '-r', GCS_CHECKPOINT, LOCAL_MERGED], check=True)
print(f'Downloaded to {LOCAL_MERGED}')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL  = 'google/gemma-4-1b-it'
LOCAL_MERGED = '/tmp/seesaw-gemma4-merged'
LOCAL_FINAL  = '/tmp/seesaw-gemma4-final'

# Merge LoRA adapters into base weights
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, LOCAL_MERGED)
merged = model.merge_and_unload()
merged.save_pretrained(LOCAL_FINAL)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.save_pretrained(LOCAL_FINAL)
print(f'Merged model saved to {LOCAL_FINAL}')

In [ ]:
import subprocess

LOCAL_FINAL   = '/tmp/seesaw-gemma4-final'
GGUF_F16      = '/tmp/seesaw-gemma4-f16.gguf'
GGUF_Q4KM     = '/tmp/seesaw-gemma4-1b-q4km.gguf'
LLAMA_DIR     = '/tmp/llama.cpp'
QUANTIZE_BIN  = f'{LLAMA_DIR}/build/bin/llama-quantize'

# Convert to GGUF float16
subprocess.run(['python', f'{LLAMA_DIR}/convert_hf_to_gguf.py',
                LOCAL_FINAL, '--outfile', GGUF_F16, '--outtype', 'f16'], check=True)

# Quantise to Q4_K_M
subprocess.run([QUANTIZE_BIN, GGUF_F16, GGUF_Q4KM, 'Q4_K_M'], check=True)

import os
size_mb = os.path.getsize(GGUF_Q4KM) / 1024 / 1024
print(f'GGUF Q4_K_M size: {size_mb:.0f} MB (expected ~800 MB)')

In [ ]:
# Validation: run 3 sample inferences
import subprocess

LLAMA_CLI = '/tmp/llama.cpp/build/bin/llama-cli'
GGUF_Q4KM = '/tmp/seesaw-gemma4-1b-q4km.gguf'

VALIDATION_PROMPT = (
    '<bos><start_of_turn>user\n'
    'Child: Vihas, age 5. Objects: teddy_bear, book. Scene: living_room. Continue the story.'
    '<end_of_turn>\n<start_of_turn>model\n'
)

result = subprocess.run(
    [LLAMA_CLI, '-m', GGUF_Q4KM, '-p', VALIDATION_PROMPT, '-n', '200', '--temp', '0.8'],
    capture_output=True, text=True
)
print('Validation output:')
print(result.stdout[-500:])  # last 500 chars

# Verify JSON parseable
import json, re
match = re.search(r'\{.*\}', result.stdout, re.DOTALL)
if match:
    parsed = json.loads(match.group())
    assert 'story_text' in parsed and 'question' in parsed
    print('✓ JSON output is valid and contains required fields')
else:
    print('⚠ JSON not found in output — check model output above')

In [ ]:
# Upload to GCS
import subprocess

GGUF_Q4KM  = '/tmp/seesaw-gemma4-1b-q4km.gguf'
GCS_DEST   = 'gs://seesaw-models/seesaw-gemma4-1b-q4km.gguf'

subprocess.run(['gsutil', 'cp', GGUF_Q4KM, GCS_DEST], check=True)
print(f'Uploaded to {GCS_DEST}')
print('Test /model/latest endpoint to confirm signed URL returns the new file.')